In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super(MLP, self).__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, output_dim))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

def train_model(model, train_loader, val_loader, lr=0.01, epochs=10):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

        if (epoch + 1) % 1 == 0:
            print(f'Epoch {epoch+1}/{epochs}, Train Loss: {train_loss/len(train_loader):.6f}, Val Loss: {val_loss/len(val_loader):.6f}')

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

input_dim = 28 * 28  # MNIST image size
hidden_dims = [128, 64]
output_dim = 10  # 10 classes (0-9)
lr = 0.001
epochs = 100

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Load MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
val_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

model = MLP(input_dim, hidden_dims, output_dim).to(device)

# Move data to device during training
def train_model(model, train_loader, val_loader, device, lr=0.01, epochs=10):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            inputs = inputs.view(inputs.size(0), -1)  # Flatten images
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                inputs = inputs.view(inputs.size(0), -1)  # Flatten images
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

        if (epoch + 1) % 1 == 0:
            print(f'Epoch {epoch+1}/{epochs}, Train Loss: {train_loss/len(train_loader):.6f}, Val Loss: {val_loss/len(val_loader):.6f}')

train_model(model, train_loader, val_loader, device, lr, epochs)

Using device: cuda
Epoch 1/100, Train Loss: 0.279791, Val Loss: 0.129899
Epoch 2/100, Train Loss: 0.113590, Val Loss: 0.102289
Epoch 3/100, Train Loss: 0.078960, Val Loss: 0.075734
Epoch 4/100, Train Loss: 0.059520, Val Loss: 0.078755
Epoch 5/100, Train Loss: 0.048529, Val Loss: 0.075620
Epoch 6/100, Train Loss: 0.038244, Val Loss: 0.074488
Epoch 7/100, Train Loss: 0.033887, Val Loss: 0.081354
Epoch 8/100, Train Loss: 0.029289, Val Loss: 0.082506
Epoch 9/100, Train Loss: 0.025692, Val Loss: 0.078668
Epoch 10/100, Train Loss: 0.021242, Val Loss: 0.091757
Epoch 11/100, Train Loss: 0.019513, Val Loss: 0.088354
Epoch 12/100, Train Loss: 0.018423, Val Loss: 0.103179
Epoch 13/100, Train Loss: 0.017455, Val Loss: 0.108669
Epoch 14/100, Train Loss: 0.016415, Val Loss: 0.110988
Epoch 15/100, Train Loss: 0.013834, Val Loss: 0.107404
Epoch 16/100, Train Loss: 0.015917, Val Loss: 0.097087
Epoch 17/100, Train Loss: 0.011930, Val Loss: 0.109095
Epoch 18/100, Train Loss: 0.014668, Val Loss: 0.137465


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw

# 配置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 创建自定义图片并测试
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('自定义手绘数字与模型预测', fontsize=14)

predictions = []

for idx in range(10):
    # 创建空白图片
    img = Image.new('L', (28, 28), color=255)  # 白色背景
    draw = ImageDraw.Draw(img)
    
    # 简单绘制数字
    if idx == 0:
        draw.ellipse([5, 5, 23, 23], outline=0, width=2)  # 画圆
    elif idx == 1:
        draw.line([(14, 5), (14, 23)], fill=0, width=2)  # 竖线
    elif idx == 2:
        draw.line([(5, 8), (23, 8), (23, 15), (5, 15), (5, 23), (23, 23)], fill=0, width=2)  # 2形
    elif idx == 3:
        draw.line([(5, 5), (23, 5), (23, 15), (5, 15), (23, 15), (23, 23), (5, 23)], fill=0, width=2)  # 3形
    elif idx == 4:
        draw.line([(23, 5), (5, 15), (23, 15), (23, 23)], fill=0, width=2)  # 4形
    elif idx == 5:
        draw.line([(23, 5), (5, 5), (5, 15), (23, 15), (23, 23), (5, 23)], fill=0, width=2)  # 5形
    elif idx == 6:
        draw.ellipse([5, 5, 23, 23], outline=0, width=2)  # 6形
        draw.line([(5, 15), (23, 15)], fill=0, width=2)
    elif idx == 7:
        draw.line([(5, 5), (23, 5), (15, 23)], fill=0, width=2)  # 7形
    elif idx == 8:
        draw.ellipse([5, 5, 23, 15], outline=0, width=2)  # 8形
        draw.ellipse([5, 15, 23, 23], outline=0, width=2)
    elif idx == 9:
        draw.ellipse([5, 5, 23, 15], outline=0, width=2)  # 9形
        draw.line([(23, 15), (23, 23)], fill=0, width=2)
    
    # 转换为tensor
    img_array = np.array(img) / 255.0  # 归一化
    img_tensor = torch.tensor(img_array, dtype=torch.float32).unsqueeze(0)  # [1, 28, 28]
    
    # 模型预测
    model.eval()
    with torch.no_grad():
        img_input = img_tensor.view(1, -1).to(device)  # Flatten并移到device
        output = model(img_input)
        _, predicted = torch.max(output, 1)
        confidence = torch.softmax(output, 1).max().item()
    
    predictions.append((idx, predicted.item(), confidence))
    
    # 显示图片
    row = idx // 5
    col = idx % 5
    axes[row, col].imshow(img_array, cmap='gray')
    axes[row, col].set_title(f'绘制: {idx}\n预测: {predicted.item()} ({confidence:.1%})', fontsize=10)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

# 显示预测结果
print('\n预测结果汇总:')
print('绘制的数字 | 模型预测 | 置信度')
print('-' * 35)
for draw_num, pred_num, conf in predictions:
    status = '✓' if draw_num == pred_num else '✗'
    print(f'{draw_num:^8} | {pred_num:^8} | {conf:>6.1%} {status}')